# Associations that change by period

A pooled correlation can hide a relationship that reverses sign over time.
The `period_associations` pass recomputes numeric-pair Pearson correlations
within periods and reports material differences. A finding is
`associations/period_association`; it describes association, not the cause
of the change.

## Periods and candidate pairs

The pass uses the first date/datetime column. It compares calendar years
when two to twelve years each contain at least 30 rows; otherwise it tries
a two-way split at the median date. Without a usable temporal column, an
integer column named `year` can define periods when its values lie in
1900–2100 and there are at most 15 distinct years.

At most 20,000 rows are used at standard and 50,000 at deep. Correlations
use complete pairs within each period and require at least 30 evaluated
pairs. If there are at most 2,048 possible numeric pairs, every pair is
considered so a sign flip need not survive the pooled dependence screen.
Above that width, candidates come from that screen; coverage discloses the
restricted search.

## The reported contrast

For each candidate, select the periods with the smallest and largest
correlations. A finding requires a difference in r of at least 0.3 and at
least one period with |r| at least 0.3. It also requires a Fisher-transform
difference of at least three standard errors:

```text
z_difference = (atanh(r_max) - atanh(r_min))
               / sqrt(1 / (n_max - 3) + 1 / (n_min - 3))
```

The calculation clips correlations slightly inside ±1 to keep the
transform finite. At most ten findings are retained. The contrast is
selected across periods and pairs; it is a diagnostic screen, not a
multiple-testing-adjusted p-value.

## A sign flip hidden by pooling


In [ ]:
import numpy as np
import polars as pl
import onceover
from great_tables import GT, loc, style

rng = np.random.default_rng(19)
x = rng.uniform(-1, 1, 800)
year = np.repeat([2020, 2021], 400)
y = np.where(year == 2020, x, -x) + rng.normal(0, 0.05, len(x))
df = pl.DataFrame({"year": year, "x": x, "y": y})
report = onceover.profile(df, detectors=["period_associations"])
findings = report.select(category="associations", type="period_association")
assert any(set(f["subject"]["columns"]) == {"x", "y"} for f in findings)
print("Pooled Pearson:", df.select(pl.corr("x", "y")).item())
for finding in findings:
    print(finding["claim"])
    metrics = finding["evidence"][0]["metrics"]
    periods = pl.DataFrame(metrics["periods"])
(GT(periods)
 .tab_header(title="The same two columns, read by period")
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


In [ ]:
import altair as alt
(alt.Chart(df).mark_circle(size=18, opacity=0.35)
 .encode(x="x:Q", y="y:Q", color=alt.Color("year:N", title="Period"),
         tooltip=["year:N", "x:Q", "y:Q"])
 .properties(title="Pooling conceals two opposite associations", width=500, height=300)
 .interactive())


## Read the periods before the headline

The finding's definition names the period column and labels. Its evidence
contains each period's row count and Pearson coefficient, the pooled
coefficient, the extreme-period contrast, and the Fisher diagnostic.
Those records let you check whether a change reflects a reversal, a loss
of strength, or one small eligible period.

Period boundaries are chosen from the time axis, not fitted as change
points. Within-period mixing, nonlinear dependence with low Pearson
correlation, sparse periods, and candidates omitted at large width can
hide structure. Use [temporal breaks](temporal_analysis.ipynb) to locate a
change in a series and [dependence measures](dependence_measures.ipynb)
for nonlinear pairwise diagnostics.
